In [36]:

%matplotlib inline
import jax.numpy as jnp
from jax import value_and_grad
from jax import grad
from jax import random
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as snb
import numpy as np
from scipy.stats import norm as norm_dist

###Bayesian ML functions###

from bayesian_ml import *
from packages.BayesianLinearRegression import BayesianLinearRegression
from packages.LogisticRegression import LogisticRegression
from packages.Grid2D import Grid2D
from packages.LaplaceApproximation import LaplaceApproximation
from packages.PosteriorPredictiveDistribution import PosteriorPredictiveDistribution
from packages.Hyperparameters import Hyperparameters
from packages.StationaryIsotropicKernel import StationaryIsotropicKernel
from packages.GaussianProcessRegression import GaussianProcessRegression
from packages.BayesianLinearSoftmax import BayesianLinearSoftmax
from packages.metropolis import metropolis

from IPython.display import Markdown, display


###Distributions###
from scipy.stats import multivariate_normal as mvn
from scipy.stats import poisson

from packages.util_funs import sigmoid
from packages.util_funs import log_npdf

snb.set_theme(font_scale=1.25)
key = random.key(seed = 0)

In [ ]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

probit = lambda x: norm_dist.cdf(x)

# Part 2:

## Question 2.1

The likelihood is given by

$p(y|f)=N(f, \sigma^2 I)$

## Question 2.2

As the gaussian process has a prior mean of 0, the prior predictive distribution is given by

$p(y^*| x_* = 1) = N(y^*|0, \kappa^2 + \sigma^2)$

## Question 2.3

For the posterior predictive distribution - $p(y^{*} | \bf{y}, x^{*})$ we have that

$$p(y^{*} | \bf{y}) = \mathcal{N}(y^{*} | \mu_{y^{*} | \bf{y}}, \sigma^{2}_{y^{*} | \bf{y}})$$
where

$$\mu_{y^* \mid y} = \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{y}$$
and
$$\sigma^2_{y^* \mid y} = c - \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{k}^T$$


In [53]:
#Define training data
xtrain = jnp.array([-2.17, 1.99 ,0.57,-3.01,-1.16, 3.30,-4.85,-0.86])[:, None]

#Define training targets
ytrain = jnp.array([0.88, 0.46,-0.06, 0.98, 0.45, 0.88,-0.66, 0.05])[:, None]

#Define test data
x_star = jnp.array([227])[:, None]

#Define design matrix 
def design_matrix(x):
    return jnp.column_stack((jnp.ones(len(x)), x))

kappa = 0.7
lengthscale = jnp.sqrt(2) / 2
sigma = 1/5

kernel = lambda x, xs: kappa**2*jnp.exp(-0.5*(x-xs)**2/lengthscale**2)

def kernel_func(x,xs):
    return kernel(x.T,xs)

k = kernel_func(x_star, xtrain)
K = kernel_func(xtrain, xtrain)
Kstar = kernel_func(x_star, x_star)
c = Kstar + jnp.eye(len(x_star)) * sigma**2

C = K + jnp.eye(len(xtrain)) * sigma**2
post_mu_y = jnp.dot(k.T,jnp.linalg.solve(C, ytrain))
post_var_y = c - jnp.dot(k.T,jnp.linalg.solve(C, k))

display(Markdown(rf"The predictive posterior distribution is $p(y_* \mid x_*, \bf{{y}}) = \mathcal{{N}}(y_* \mid \mu_{{y^{{*}} | y}}, \sigma_{{y^{{*}} | y}}^2)$ where $\mu_* = {to_latex_matrix(post_mu_y)}$ and $\sigma_*^2 = {to_latex_matrix(post_var_y)}$"))

The predictive posterior distribution is $p(y_* \mid x_*, \bf{y}) = \mathcal{N}(y_* \mid \mu_{y^{*} | y}, \sigma_{y^{*} | y}^2)$ where $\mu_* = \begin{bmatrix}0.00\end{bmatrix}$ and $\sigma_*^2 = \begin{bmatrix}0.53\end{bmatrix}$

## Question 2.4

The marginal likelihood is given by $$p(\bf{y}| \theta) = \mathcal{N}(\bf{y} | \bf{0}, \beta^{-1} \bf{I} + \bf{K})$$

In [54]:
ml_var = sigma**2 * jnp.eye(len(xtrain)) + K
ml_mu = jnp.zeros(len(xtrain))


display(Markdown(
    rf"The marginal likelihood is $p(\mathbf{{y}} \mid \theta) = "
    rf"\mathcal{{N}}(\mathbf{{y}} \mid \mathbf{{0}}, \beta^{{-1}}\mathbf{{I}} + \mathbf{{K}})$"
    rf" where $\beta^{{-1}}\mathbf{{I}} + \mathbf{{K}} = {to_latex_matrix(ml_var)}$"
))

The marginal likelihood is $p(\mathbf{y} \mid \theta) = \mathcal{N}(\mathbf{y} \mid \mathbf{0}, \beta^{-1}\mathbf{I} + \mathbf{K})$ where $\beta^{-1}\mathbf{I} + \mathbf{K} = \begin{bmatrix}0.53 & 0.00 & 0.00 & 0.24 & 0.18 & 0.00 & 0.00 & 0.09 \\ 0.00 & 0.53 & 0.07 & 0.00 & 0.00 & 0.09 & 0.00 & 0.00 \\ 0.00 & 0.07 & 0.53 & 0.00 & 0.02 & 0.00 & 0.00 & 0.06 \\ 0.24 & 0.00 & 0.00 & 0.53 & 0.02 & 0.00 & 0.02 & 0.00 \\ 0.18 & 0.00 & 0.02 & 0.02 & 0.53 & 0.00 & 0.00 & 0.45 \\ 0.00 & 0.09 & 0.00 & 0.00 & 0.00 & 0.53 & 0.00 & 0.00 \\ 0.00 & 0.00 & 0.00 & 0.02 & 0.00 & 0.00 & 0.53 & 0.00 \\ 0.09 & 0.00 & 0.06 & 0.00 & 0.45 & 0.00 & 0.00 & 0.53\end{bmatrix}$

In [55]:
const_term = -len(xtrain) /2 * jnp.log(2*jnp.pi)
log_det_term = -1/2 * jnp.log(jnp.linalg.det(C)) 
quad_term = -1/2 * jnp.dot(ytrain.T, jnp.linalg.solve(C, ytrain))

lm_likelihood = const_term + log_det_term + quad_term
display(Markdown(
    rf"The log marginal likelihood is "
    rf"$\ln p(\mathbf{{y}} \mid \boldsymbol{{\theta}}) = "
    rf"-\frac{{N}}{{2}}\ln(2\pi) - \frac{{1}}{{2}}\ln|\beta^{{-1}}\mathbf{{I}} + \mathbf{{K}}| - \frac{{1}}{{2}}\mathbf{{y}}^T(\beta^{{-1}}\mathbf{{I}} + \mathbf{{K}})^{{-1}}\mathbf{{y}}$"
    rf" $= {const_term:.4f} + ({log_det_term:.4f}) + ({to_latex_matrix(quad_term)}) = {to_latex_matrix(lm_likelihood)}$"
))

The log marginal likelihood is $\ln p(\mathbf{y} \mid \boldsymbol{\theta}) = -\frac{N}{2}\ln(2\pi) - \frac{1}{2}\ln|\beta^{-1}\mathbf{I} + \mathbf{K}| - \frac{1}{2}\mathbf{y}^T(\beta^{-1}\mathbf{I} + \mathbf{K})^{-1}\mathbf{y}$ $= -7.3515 + (3.4183) + (\begin{bmatrix}-2.72\end{bmatrix}) = \begin{bmatrix}-6.65\end{bmatrix}$

## Question 2.5

The posterior predictive distribution is given by

$$p(f^{*} | \bf{y}) = \mathcal{N}(f^{*} | \mu_{f^{*} | \bf{y}}, \sigma^{2}_{f^{*} | \bf{y}})$$
where

$$\mu_{f^* \mid y} = \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{y}$$
and
$$\sigma^2_{f^* \mid y} = \bf{K_{*}} - \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{k}^T$$


In [59]:
#Define test data
x_star = jnp.array([1])[:, None]

k = kernel_func(x_star, xtrain)
K = kernel_func(xtrain, xtrain)
Kstar = kernel_func(x_star, x_star)
c = Kstar + jnp.eye(len(x_star)) * sigma**2

C = K + jnp.eye(len(xtrain)) * sigma**2
post_mu_f = jnp.dot(k.T,jnp.linalg.solve(C, ytrain))
post_var_f = Kstar - jnp.dot(k.T,jnp.linalg.solve(C, k))

display(Markdown(rf"The posterior predictive distribution is $p(f_* \mid x_*, \bf{{y}}) = \mathcal{{N}}(f_* \mid \mu_{{f^{{*}} | y}}, \sigma_{{f^{{*}} | y}}^2)$ where $\mu_* = {to_latex_matrix(post_mu_f)}$ and $\sigma_*^2 = {to_latex_matrix(post_var_f)}$"))

The posterior predictive distribution is $p(f_* \mid x_*, \bf{y}) = \mathcal{N}(f_* \mid \mu_{f^{*} | y}, \sigma_{f^{*} | y}^2)$ where $\mu_* = \begin{bmatrix}0.07\end{bmatrix}$ and $\sigma_*^2 = \begin{bmatrix}0.14\end{bmatrix}$

In [60]:
C = K + jnp.eye(len(xtrain)) * sigma**2
post_mu_y = jnp.dot(k.T,jnp.linalg.solve(C, ytrain))
post_var_y = c - jnp.dot(k.T,jnp.linalg.solve(C, k))

display(Markdown(rf"The predictive posterior distribution is $p(y_* \mid x_*, \bf{{y}}) = \mathcal{{N}}(y_* \mid \mu_{{y^{{*}} | y}}, \sigma_{{y^{{*}} | y}}^2)$ where $\mu_* = {to_latex_matrix(post_mu_y)}$ and $\sigma_*^2 = {to_latex_matrix(post_var_y)}$"))

The predictive posterior distribution is $p(y_* \mid x_*, \bf{y}) = \mathcal{N}(y_* \mid \mu_{y^{*} | y}, \sigma_{y^{*} | y}^2)$ where $\mu_* = \begin{bmatrix}0.07\end{bmatrix}$ and $\sigma_*^2 = \begin{bmatrix}0.18\end{bmatrix}$

## Question 2.6

If a kernel is a **stationary** kernel, it means that the covariance function only depends on the difference between two points, i.e. $\tau = \mathbf{x}_n - \mathbf{x}_m$ for two points in the input space $\mathbf{x}_n, \mathbf{x}_m \in \mathbb{R}^D$

For a kernel to be isotropic the covariance function only depends on the distance between any two inputs, i.e. $||\tau||_2 = ||\mathbf{x}_n - \mathbf{x}_m||_2$.

Since $k_2$ cannot be written to depend only on the difference between two input points it is neither stationary, nor isotropic.

## Question 2.7

In [61]:
#Parameters
c1 = 1
c2 = 1
lengthscale = 1/jnp.sqrt(2)

#Kernel
kernel2 = lambda x, xs: c1 * (1 + jnp.abs(x - xs) / (2*lengthscale**2))**(-1) + c2*x*xs

def kernel_func2(x,xs):
    return kernel2(x.T,xs)

#New xstar
xstar = jnp.array([-1])[:,None]

kernel_func2(xtrain,xstar)

Array([[ 2.63082949, -1.73937343, -0.18089494,  3.34222591,  2.02206897,
        -3.11132075,  5.05618557,  1.73719298]], dtype=float64)

# Part 3:

## Question 3.2

In [19]:
def design_matrix(x):
    return jnp.column_stack((jnp.ones(len(x)), x, x**2))

Xtest = jnp.array([-3])
W_map = jnp.array([2.647, -1.688, -0.596])
Phi_test = design_matrix(Xtest)

mu = Phi_test @ W_map.T
dist = sigmoid(mu)
display(Markdown(rf"The predictive distribution is $p(y^*| y, x_* = -3) = Ber(y^*|{dist[0]:.2f})$"))

The predictive distribution is $p(y^*| y, x_* = -3) = Ber(y^*|0.91)$

## Question 3.3

In [27]:
m_w0 = W_map[0]
s_w0 = jnp.sqrt(3)
credibility_width = 90
variable_name = "w0"

cred_90 = norm.interval(credibility_width/100, m_w0, s_w0)

display(Markdown(rf"The {credibility_width} \% credibility interval is given by $p({cred_90[0]:.2f} < {variable_name} < {cred_90[1]:.2f}|y) = {credibility_width/100}$"))

The 90 \% credibility interval is given by $p(-0.20 < w0 < 5.50|y) = 0.9$

## Question 3.4

In [40]:
S = jnp.array([[3, -0.39, -0.3], [-0.39, 1.55, 0.37], [-0.3, 0.37, 0.14]])
approx_samples = random.multivariate_normal(key, W_map, S, shape=(1000, ))

m = jnp.mean(Phi_test @ approx_samples.T)
v = jnp.var(Phi_test @ approx_samples.T)

p = probit(m/jnp.sqrt(8/jnp.pi + v))

m, v, p

(Array(2.38245936, dtype=float64),
 Array(5.27167314, dtype=float64),
 np.float64(0.8029113801408254))

## Question 3.5

In [47]:
def compute_expected_utility(U, phat):
    """ computes the expected utility for a multi-class classification problem with K classes for utility matrix U and posterior predictive probabilities phat 
        
        Arguments
        U               --      Utility matrix (shape: [K x K])
        phat            --      Posterior predictive probabilities (shape: [P x K]), where P is the number of prediction points

        expected_util   --      Expected utility for each class for each point in phat (shape: P x K)           
           """
    

    expected_util = phat@U  

    return expected_util

U = jnp.array([[2, 1], [1, 2]])
p_all = jnp.array([1-0.129, 0.129])

expected_util = compute_expected_utility(U, p_all)

zero_index = True
display(Markdown(rf"The expected utility is"))
for i, p in enumerate(expected_util.flatten()):
    i -= 1 if zero_index else 0
    display(Markdown(rf"$\mathbb{{E}}_{{p(y^*|\mathbf{{y}}, \mathbf{{x}}^*)}}\left[\mathcal{{U}}(y^*, \hat{{y}}={i+1})\right]  = {p:.2f}$"))

optimal_decision = jnp.argmax(expected_util) if zero_index else jnp.argmax(expected_util)+1

display(Markdown(rf"The optimal decision is thus $y^*={optimal_decision}$"))

The expected utility is

$\mathbb{E}_{p(y^*|\mathbf{y}, \mathbf{x}^*)}\left[\mathcal{U}(y^*, \hat{y}=0)\right]  = 1.87$

$\mathbb{E}_{p(y^*|\mathbf{y}, \mathbf{x}^*)}\left[\mathcal{U}(y^*, \hat{y}=1)\right]  = 1.13$

The optimal decision is thus $y^*=0$